# 01 — Data Collection
Load, inspect, and validate recorded reaching trials.

**CSV schema expected:** one row per timestep, columns:
`time, q0, q1, q2, q3, q4, q5, q6, target_x, target_y, target_z`

| Index | Joint |
|-------|-------|
| q0 | shoulder flexion/extension |
| q1 | shoulder abduction/adduction |
| q2 | shoulder int/ext rotation |
| q3 | elbow flexion/extension |
| q4 | forearm pronation/supination |
| q5 | wrist flexion/extension |
| q6 | wrist radial/ulnar deviation |

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path('../data')
JOINT_COLS = [f'q{i}' for i in range(7)]
TARGET_COLS = ['target_x', 'target_y', 'target_z']

## Load a trial

In [ ]:
# TODO: replace with your actual file
trial_path = DATA_DIR / 'trial_001.csv'

df = pd.read_csv(trial_path)
print(df.shape)
df.head()

## Quick sanity checks

In [ ]:
assert set(JOINT_COLS + TARGET_COLS + ['time']).issubset(df.columns), \
    f"Missing columns: {set(JOINT_COLS + TARGET_COLS + ['time']) - set(df.columns)}"
assert df.isnull().sum().sum() == 0, "NaNs found in data"
print("All checks passed.")
print(f"Duration: {df['time'].iloc[-1]:.2f}s  |  Samples: {len(df)}  |  Hz: {1/(df['time'].diff().median()):.1f}")

## Plot joint trajectories

In [ ]:
fig, axes = plt.subplots(7, 1, figsize=(10, 14), sharex=True)
labels = [
    'Shoulder flex/ext', 'Shoulder abd/add', 'Shoulder int/ext rot',
    'Elbow flex/ext', 'Forearm pron/sup',
    'Wrist flex/ext', 'Wrist rad/uln dev',
]
for ax, col, label in zip(axes, JOINT_COLS, labels):
    ax.plot(df['time'], np.degrees(df[col]))
    ax.set_ylabel(f'{label}\n(deg)')
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Joint trajectories — healthy arm', y=1.01)
plt.tight_layout()
plt.show()

## Build dataset dict (format expected by `src/optimize.py`)

In [ ]:
def load_trial(path: Path) -> dict:
    df = pd.read_csv(path)
    return {
        't':      df['time'].to_numpy(),
        'q_ref':  df[JOINT_COLS].to_numpy(),
        'target': df[TARGET_COLS].iloc[-1].to_numpy(),  # target = final position
    }

# dataset = [load_trial(p) for p in sorted(DATA_DIR.glob('*.csv'))]
# print(f"Loaded {len(dataset)} trials")